# Day 4 · Exercise 2: JSON Mode

**What you'll build:** `extract_json_reliable` — add `format="json"` to guarantee parseable output.

**Why it matters:** One parameter change eliminates the `JSONDecodeError` problem entirely. Ollama constrains the model's token sampling so it can only produce valid JSON — no prose, no markdown, no surprises. This is layer 2 of the three-layer reliability system.

## Your Implementation

In [ ]:
import ollama
import json

MODEL = "llama3.2"

def extract_json_reliable(text: str) -> dict:
    """Extract information from text using JSON mode.

    Use ollama.chat with:
    - A system prompt that specifies the exact keys to return:
      name (string), age (integer), city (string)
    - format="json" to guarantee valid JSON syntax

    Parse the response with json.loads() and return the result.

    Args:
        text: A sentence or paragraph about a person.

    Returns:
        A dict with keys: name, age, city.

    Example:
        extract_json_reliable("Alice is 30 and lives in Cape Town.")
        -> {"name": "Alice", "age": 30, "city": "Cape Town"}
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work

Run the cell below — it runs 4 automated checks and shows ✅ / ❌ for each.

In [ ]:
_PASS, _FAIL = '✅', '❌'

def _ollama_running():
    try:
        import urllib.request  # stdlib — no install needed
        urllib.request.urlopen('http://localhost:11434/api/tags', timeout=3)
        return True
    except Exception:
        return False

def _run_checks():
    score, total = 0, 4

    # Check 1: function exists and is callable
    try:
        assert callable(extract_json_reliable), 'extract_json_reliable is not defined'
        print(f'{_PASS} Check 1/{total}: function exists and is callable')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 1/{total}: {e}')
        return

    # Check 2: Ollama is running
    if not _ollama_running():
        print(f'{_FAIL} Check 2/{total}: Ollama server is not running')
        print('  → macOS: open the Ollama app · Linux/Windows: run ollama serve')
        return
    print(f'{_PASS} Check 2/{total}: Ollama server is reachable')
    score += 1

    # Check 3: returns a dict (json.loads must succeed)
    result = None
    try:
        result = extract_json_reliable("Alice is 30 years old and lives in Cape Town.")
        assert isinstance(result, dict), f'expected dict, got {type(result).__name__}'
        print(f'{_PASS} Check 3/{total}: returned a dict (JSON mode is working)')
        score += 1
    except json.JSONDecodeError as e:
        print(f'{_FAIL} Check 3/{total}: JSONDecodeError — did you add format="json"?')
        print(f'   Error: {e}')
    except AssertionError as e:
        print(f'{_FAIL} Check 3/{total}: {e}')
    except Exception as e:
        print(f'{_FAIL} Check 3/{total}: call failed — {e}')
        return

    if result is None:
        return

    # Check 4: dict contains a "name" key
    try:
        assert 'name' in result, f'key "name" missing from result — got keys: {list(result.keys())}'
        print(f'{_PASS} Check 4/{total}: dict contains "name" key (system prompt was followed)')
        score += 1
    except AssertionError as e:
        print(f'{_FAIL} Check 4/{total}: {e}')
        print('  → Update your system prompt to specify the exact keys: name, age, city')

    print()
    if score == total:
        print('=' * 52)
        print(f'  {_PASS}  Exercise 2 complete! {total}/{total} checks passed.')
        print('=' * 52)
    else:
        print(f'  {score}/{total} passed. Keep going!')

_run_checks()

## Bonus Challenge

Run `extract_json_reliable` ten times with the same input and check consistency:

```python
text = "Bob is a 28-year-old software engineer in Johannesburg."
results = [extract_json_reliable(text) for _ in range(10)]
key_sets = [tuple(sorted(r.keys())) for r in results]
print("Unique key sets:", set(key_sets))  # ideally just one
print("Name values:", [r.get('name') for r in results])
```

Compare to Exercise 1. JSON mode guarantees parseable output every time. The keys should be much more consistent thanks to your system prompt — though exact values may vary slightly by run.

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
import ollama
import json

MODEL = "llama3.2"

def extract_json_reliable(text: str) -> dict:
    response = ollama.chat(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "Extract information and return a JSON object with exactly "
                    "these keys: name (string), age (integer), city (string). "
                    "Return only the JSON object."
                ),
            },
            {"role": "user", "content": text},
        ],
        format="json",
    )
    return json.loads(response["message"]["content"])
```

**Why format="json" works:** Ollama uses grammar-constrained token sampling — at each step it only allows tokens that keep the output valid JSON. The model literally cannot produce prose or a broken JSON object. `json.loads()` always succeeds. Combined with a system prompt that names the expected keys, this gives you consistent output every time.
</details>